# Text as Data — Tokenizers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/ms2a-machine-learning-practice/challenges/mlp-s7-tokenization.ipynb)

A model never reads text. It reads a sequence of integer ids, and the **tokenizer**
decides what those ids are. This notebook takes five tokenizers from the Hugging Face
Hub and compares them on three things:

1. **Their vocabulary** — what tokens they contain, and for which writing systems.
2. **A parallel corpus** — the same 960 sentences in eight languages: how many tokens
   each tokenizer needs to encode the same content.
3. **A model** — GPT-2 fed with its own tokenizer, then with another one.

No training here: everything runs on CPU in a few minutes.

## 0. Setup

In [ ]:
!pip install -q transformers sentencepiece pandas matplotlib

import html
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from transformers import AutoTokenizer
from transformers.utils import logging

logging.set_verbosity_error()

## 1. Five tokenizers

| Name | Model it belongs to | Algorithm | Trained on |
|---|---|---|---|
| `gpt2` | GPT-2 (2019) | byte-level BPE | English web text |
| `bert-base-uncased` | BERT (2018) | WordPiece, lowercased, accents stripped | English Wikipedia + books |
| `camembert-base` | CamemBERT (2019) | SentencePiece | French web text |
| `xlm-roberta-base` | XLM-R (2019) | SentencePiece | 100 languages |
| `Qwen/Qwen2.5-0.5B` | Qwen 2.5 (2024) | byte-level BPE | multilingual web text + code |

Only the tokenizer files are downloaded (a few MB each), not the models.

In [ ]:
NAMES = ["gpt2", "bert-base-uncased", "camembert-base", "xlm-roberta-base", "Qwen/Qwen2.5-0.5B"]
toks = {n: AutoTokenizer.from_pretrained(n) for n in NAMES}
COLORS = dict(zip(NAMES, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#4a3aa7"]))

for n, t in toks.items():
    print(f"{n:20s} vocabulary: {len(t):>7,} tokens")

Each tokenizer maps text to ids and back. `decode(encode(text))` returns the text,
except where the tokenizer threw information away on purpose (BERT lowercases and
removes accents).

In [ ]:
text = "Le tokenizer découpe «anticonstitutionnellement» en 2026."
for n, t in toks.items():
    ids = t.encode(text, add_special_tokens=False)
    print(f"{n:20s} {len(ids):2d} ids {ids[:8]}...  ->  {t.decode(ids)!r}")

To see the split, `show_tokens` colours each token. A `·` marks a space that belongs to
the token: most tokenizers glue the space to the **start** of the next word (`Ġ` in
GPT-2 and Qwen, `▁` in SentencePiece), BERT marks the **continuation** of a word instead (`##`).

In [ ]:
PALETTE = ["#dbeafe", "#fde2d4", "#d1f2e4", "#fdf0c8", "#e9e4fb"]

def show_tokens(text, name):
    t = toks[name]
    ids = t.encode(text, add_special_tokens=False)
    pieces = [(" " if tok.startswith("▁") else "") + t.decode([i])
              for i, tok in zip(ids, t.convert_ids_to_tokens(ids))]
    spans = "".join(
        f'<span style="background:{PALETTE[k % len(PALETTE)]};color:#111;padding:1px 2px;margin:1px;'
        f'border-radius:3px;font-family:monospace">{html.escape(p.replace(" ", "·"))}</span>'
        for k, p in enumerate(pieces))
    display(HTML(f'<div style="margin:2px 0"><b style="display:inline-block;width:170px">{name}</b>'
                 f'<span style="display:inline-block;width:70px">{len(ids)} tokens</span>{spans}</div>'))

for text in ["The unbelievably tokenized sentence.",
             "Le tokenizer découpe «anticonstitutionnellement» en 2026.",
             "Токенизатор режет текст на части.",
             "分词器把文本切成小块。"]:
    print(text)
    for n in NAMES:
        show_tokens(text, n)

**Observe.** On English, `gpt2` and `Qwen` keep `unbelievably` almost whole, while the
French `camembert-base` cuts English into short pieces. On French it is the reverse:
`camembert-base` keeps `découpe` whole where the English tokenizers cut it. On Russian and Chinese, `gpt2`
falls back to single **bytes** (a Cyrillic letter is 2 bytes in UTF-8, a Chinese
character 3); the `�` pieces are byte fragments that only form a character together.
`bert-base-uncased` and `camembert-base` have no byte fallback: an unknown character
becomes `[UNK]` / `<unk>` and is lost. `camembert-base` even swallows a whole run of
unknown characters into a single `<unk>`: the Chinese sentence becomes 2 tokens.

## 2. The vocabulary

A vocabulary is a fixed list of strings. What is in the list says what the tokenizer
was built for. We sort every token by the writing system of its letters.

In [ ]:
SCRIPTS = ["Latin", "Cyrillic", "Arabic", "Devanagari", "CJK", "digits", "byte fragment", "other"]

def script_of(s):
    if "\ufffd" in s:
        return "byte fragment"
    for ch in s:
        if ch.isalpha():
            name = unicodedata.name(ch, "")
            for key, script in [("LATIN", "Latin"), ("CYRILLIC", "Cyrillic"), ("ARABIC", "Arabic"),
                                ("DEVANAGARI", "Devanagari"), ("CJK", "CJK"), ("HIRAGANA", "CJK"),
                                ("KATAKANA", "CJK"), ("HANGUL", "CJK")]:
                if name.startswith(key):
                    return script
            return "other"
    return "digits" if any(ch.isdigit() for ch in s) else "other"

def vocab_strings(name):
    t = toks[name]
    specials = set(t.all_special_tokens)
    return [t.convert_tokens_to_string([tok]).replace("##", "").strip()
            for tok in t.get_vocab() if tok not in specials]

vocab = {n: vocab_strings(n) for n in NAMES}
share = pd.DataFrame({n: pd.Series([script_of(s) for s in v]).value_counts(normalize=True)
                      for n, v in vocab.items()}).T.reindex(columns=SCRIPTS).fillna(0)
(share * 100).round(1)

In [ ]:
SCRIPT_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#b0aea5"]
fig, ax = plt.subplots(figsize=(9, 3.2))
left = np.zeros(len(NAMES))
for script, color in zip(SCRIPTS, SCRIPT_COLORS):
    ax.barh(NAMES, share[script], left=left, color=color, label=script, edgecolor="white", linewidth=1)
    left += share[script].values
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel("share of the vocabulary")
ax.legend(ncol=4, bbox_to_anchor=(0.5, -0.3), loc="upper center", frameon=False)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
ax.set_title("Which writing systems each vocabulary contains")
plt.tight_layout(); plt.show()

**Observe.** `gpt2`, `bert-base-uncased` and `camembert-base` are almost entirely Latin:
a Russian or Chinese text has almost no whole tokens to use. `xlm-roberta-base` spreads
its 250k tokens across scripts (the grey block is the many other scripts it covers).
`Qwen` keeps a large CJK block (it is a Chinese model) but no Devanagari token at all:
Hindi will be written byte by byte.

The longest tokens show what was frequent in the training text: a tokenizer learns
whatever repeated, including web boilerplate.

In [ ]:
for n in ["gpt2", "camembert-base", "Qwen/Qwen2.5-0.5B"]:
    longest = sorted(set(vocab[n]), key=len, reverse=True)[:6]
    print(f"{n:20s}", [s[:40] for s in longest])

**Numbers** are a design choice too. GPT-2 learned whatever digit groups were frequent,
so the same number is cut differently depending on its value. Qwen splits every number
into single digits: longer sequences, but a consistent place value, which helps arithmetic.

In [ ]:
for n in ["gpt2", "Qwen/Qwen2.5-0.5B"]:
    n_digit_tokens = sum(s.isdigit() for s in vocab[n])
    print(f"{n:20s} {n_digit_tokens:5d} all-digit tokens")
    for number in ["1999", "2026", "12345", "3.14159"]:
        print(f"{'':26s}{number:>8s} -> {toks[n].tokenize(number)}")

## 3. A parallel corpus: the same content, eight languages

[WMT24++](https://huggingface.co/datasets/google/wmt24pp) gives the **same** English
source sentences translated into 55 languages by professionals. Because the content is
identical, the number of tokens differs only because of the tokenizer.

In [ ]:
LANGS = {"en": None, "fr": "fr_FR", "de": "de_DE", "ru": "ru_RU",
         "ar": "ar_EG", "hi": "hi_IN", "zh": "zh_CN", "sw": "sw_KE"}
URL = "hf://datasets/google/wmt24pp/en-{}.jsonl"

frames = {code: pd.read_json(URL.format(loc), lines=True) for code, loc in LANGS.items() if loc}
keep = ~frames["fr"]["is_bad_source"]
corpus = pd.DataFrame({"en": frames["fr"].loc[keep, "source"]})
for code, df in frames.items():
    corpus[code] = df.loc[keep, "target"]
print(len(corpus), "sentences")
corpus.iloc[3]

In [ ]:
def n_tokens(name, texts):
    return sum(len(ids) for ids in toks[name](list(texts), add_special_tokens=False)["input_ids"])

counts = pd.DataFrame({n: {lang: n_tokens(n, corpus[lang]) for lang in corpus} for n in NAMES}).T
counts

Divide each row by its English count: the **token premium** of a language, i.e. how
many times more tokens the tokenizer needs to say the same thing than in English.

In [ ]:
premium = counts.div(counts["en"], axis=0)

fig, ax = plt.subplots(figsize=(9, 3.4))
im = ax.imshow(premium.values, cmap="Blues", vmin=1, vmax=premium.values.max())
ax.set_xticks(range(len(premium.columns)), premium.columns)
ax.set_yticks(range(len(NAMES)), NAMES)
for i in range(premium.shape[0]):
    for j in range(premium.shape[1]):
        v = premium.values[i, j]
        ax.text(j, i, f"{v:.1f}×", ha="center", va="center", fontsize=9,
                color="white" if v > 0.55 * premium.values.max() else "#111")
ax.set_title("Tokens needed for the same content, relative to English")
plt.tight_layout(); plt.show()

**Observe.** With `gpt2`, a Hindi sentence costs several times more tokens than its
English translation, and French already costs almost twice as much. `xlm-roberta-base`
stays close to 1× everywhere; `Qwen` does too, except on Hindi, which its vocabulary
does not cover (section 2). This is
not cosmetic: a model has a fixed **context window** (1,024 tokens for GPT-2), costs
compute **per token**, and paid APIs bill **per token**. The same document is shorter,
cheaper and fits more context in some languages than in others (Petrov et al., 2023,
*Language Model Tokenizers Introduce Unfairness Between Languages*).

Some cells look *better* than English: `camembert-base` on Chinese (0.2×) and Arabic (0.9×),
`bert-base-uncased` on Chinese (1.4×, less than on French). They are not: count what
those tokens are.

In [ ]:
def unk_rate(name, texts):
    ids = [i for row in toks[name](list(texts), add_special_tokens=False)["input_ids"] for i in row]
    return np.mean(np.array(toks[name].convert_ids_to_tokens(ids)) == toks[name].unk_token)

unk = pd.DataFrame({n: {lang: unk_rate(n, corpus[lang]) for lang in corpus} for n in NAMES}).T
(unk * 100).round(1).astype(str) + " %"

A token that is `[UNK]` carries no information: every unknown character collapses onto
the same id. Two thirds of BERT's Chinese tokens are `[UNK]`, and CamemBERT loses 40–46 % of its
tokens to `<unk>` on Russian, Arabic, Hindi and Chinese, each one standing for a whole
run of characters. A short sequence is only good news if the tokens still carry the text. Byte-level tokenizers (`gpt2`, `Qwen`) are at 0 % by construction; a
tokenizer without byte fallback silently deletes what it cannot represent.

### Tokens per word

For languages written with spaces we can also count **tokens per word** (the
*fertility*). Near 1, most words are single tokens; higher, words are cut into pieces
whose meaning the model has to reassemble.

In [ ]:
def fertility(name, texts):
    words = [w for s in texts for w in s.split()]
    return n_tokens(name, [" " + w for w in words]) / len(words)

fert = pd.DataFrame({n: {lang: fertility(n, corpus[lang]) for lang in ["en", "fr", "de", "ru"]}
                     for n in NAMES}).T

fig, ax = plt.subplots(figsize=(9, 3.2))
width = 0.16
x = np.arange(fert.shape[1])
for k, n in enumerate(NAMES):
    ax.bar(x + (k - 2) * width, fert.loc[n], width - 0.02, color=COLORS[n], label=n)
ax.set_xticks(x, fert.columns)
ax.set_ylabel("tokens per word")
ax.axhline(1, color="#b0aea5", linewidth=1)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
ax.legend(frameon=False, fontsize=8, ncol=5, bbox_to_anchor=(0.5, -0.15), loc="upper center")
plt.tight_layout(); plt.show()
fert.round(2)

### How often each token is used

Count every token of the English side with `gpt2`. Token frequencies follow **Zipf's
law**: the frequency of the token of rank $r$ is roughly proportional to $1/r$, a
straight line on a log–log plot. A handful of tokens (`the`, `,`, `.`) make a large
share of the text, and most of the vocabulary is used rarely or never.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
for n in ["gpt2", "xlm-roberta-base"]:
    freq = Counter(i for row in toks[n](list(corpus["en"]), add_special_tokens=False)["input_ids"] for i in row)
    f = np.array(sorted(freq.values(), reverse=True))
    ax.loglog(np.arange(1, len(f) + 1), f, color=COLORS[n], linewidth=2, label=n)
    top = f[:100].sum() / f.sum()
    print(f"{n:18s} {len(freq):6,} distinct tokens used out of {len(toks[n]):,} "
          f"({len(freq) / len(toks[n]):.1%});  the 100 most frequent make {top:.0%} of the text")
    print(f"{'':18s} top 10:", [toks[n].decode([i]) for i, _ in freq.most_common(10)])
ax.set_xlabel("rank of the token"); ax.set_ylabel("occurrences")
ax.set_title("Rank–frequency of tokens on the English side (log–log)")
ax.legend(frameon=False)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

**Observe.** On an English text, the multilingual tokenizer uses a smaller fraction of
its vocabulary: most of its 250k rows serve other languages. Each row is a vector in the
model's embedding matrix, so vocabulary size trades parameters against sequence length.

### Your turn

Pick a language, a domain (Python code, chemistry SMILES, a French legal text, emoji, a
DNA sequence) and compare the five tokenizers with `show_tokens`. Which one would you
choose to build a model on that data, and why?

In [ ]:
my_text = "def f(x):\n    return x ** 2  # 🐍"  # <-- change me
for n in NAMES:
    show_tokens(my_text, n)

## 4. A model and its tokenizer

A model is trained on the ids of **one** tokenizer: row $i$ of its embedding matrix
means whatever token $i$ meant during training. Give GPT-2 the ids of another tokenizer
and every id still points to a row, just the wrong one.

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
gpt2 = toks["gpt2"]

prompt = "The capital of France is Paris. The capital of Germany is"
ids_right = gpt2.encode(prompt)
ids_wrong = toks["bert-base-uncased"].encode(prompt, add_special_tokens=False)

print("GPT-2 tokenizer ids:", ids_right)
print("BERT tokenizer ids: ", ids_wrong)
print()
print("What GPT-2 reads from its own ids:", repr(gpt2.decode(ids_right)))
print("What GPT-2 reads from BERT's ids: ", repr(gpt2.decode(ids_wrong)))

Measure it with the model's own quantity: the average next-token loss
$\mathcal{L} = -\frac{1}{T}\sum_t \log p(x_{t+1}\mid x_{\le t})$ and the perplexity
$e^{\mathcal{L}}$, the number of tokens the model hesitates between at each step.

In [ ]:
@torch.no_grad()
def loss_and_continuation(ids, n_new=8):
    x = torch.tensor([ids])
    loss = model(x, labels=x).loss.item()
    out = model.generate(x, max_new_tokens=n_new, do_sample=False, pad_token_id=gpt2.eos_token_id)
    return loss, out[0, len(ids):].tolist()

for label, ids in [("GPT-2 tokenizer", ids_right), ("BERT tokenizer", ids_wrong)]:
    loss, new = loss_and_continuation(ids)
    print(f"{label:16s} loss {loss:5.2f}   perplexity {np.exp(loss):9.0f}")
    print(f"{'':16s} continuation: {gpt2.decode(new)!r}")

With its own tokenizer, GPT-2 continues with `Berlin`. With BERT's ids it reads a
string of unrelated tokens; the perplexity jumps by orders of magnitude and the output
is meaningless. No error was raised: the ids were valid
integers. **Always load the tokenizer from the same checkpoint as the model**
(`AutoTokenizer.from_pretrained(name)` with the same `name`).

The right tokenizer is still not the whole story. The same GPT-2 on the same content in
French: every sentence of the corpus, English then French.

In [ ]:
@torch.no_grad()
def corpus_stats(texts):
    losses, n_tok = [], 0
    for s in texts:
        ids = gpt2.encode(s)[:1024]
        x = torch.tensor([ids])
        losses.append(model(x, labels=x).loss.item() * (len(ids) - 1))
        n_tok += len(ids) - 1
    return n_tok, sum(losses) / n_tok

sample = corpus.sample(100, random_state=0)
for lang in ["en", "fr"]:
    n_tok, loss = corpus_stats(sample[lang])
    n_chars = sample[lang].str.len().sum()
    print(f"{lang}: {n_tok:5d} tokens, {n_chars / n_tok:.2f} characters per token, "
          f"loss {loss:.2f} per token, {loss * n_tok / n_chars:.2f} per character")

**Observe.** On French, GPT-2 needs more tokens for the same content, and each token is
harder to predict. The tokenizer and the training data were both English; a model can
only be as good on a language as its tokenizer lets it read.

## Questions

1. Why does a byte-level tokenizer never produce `[UNK]`, and what does it pay for that?
2. `bert-base-uncased` gets a small token count on Chinese. Why is it a bad sign here?
3. You fine-tune a model on French clinical notes. Which two numbers from this notebook
   would you compute on your data before choosing the pretrained checkpoint?
4. Doubling the vocabulary shortens sequences. What does it cost in the model?